# Evaluate Models on Wind Turbine Blade Inspection

This notebook scores frontier models (Gemini, GPT, Claude), the LFM3-VL baseline and a fine-tuned LFM3-VL on the same task, so they can be compared directly. Each run writes `results/<run name>.jsonl`, and the results section at the end scores every run on the same images.

**Data:** [WTBs2025](https://doi.org/10.6084/m9.figshare.28876391) (CC BY 4.0, [Scientific Data 2026](https://doi.org/10.1038/s41597-026-07455-1)): 7,544 blade photos, mostly taken by drone, with YOLO boxes for 9 defect types. Each image shows one defect type, often several instances of it, and many defects are small relative to the image.

**Task:** given a blade photo, return every defect as `{"label", "box"}`. The prompt lists the 9 defect types by name, with no definitions or examples.

**Fairness choices:**
- **Same images:** a fixed, held-out test split. Augmented copies in the dataset are left out, as are photos that appear under two defect types, and the same split stays out of fine-tuning. Images are sent at their original resolution.
- **Same task text:** only the answer-format sentence differs. Every model answers on a 0–1000 grid, so no provider's image resizing can shift the boxes. Gemini gets its native `[ymin, xmin, ymax, xmax]` order; all other models, LFM3-VL included, answer `[x_min, y_min, x_max, y_max]`.
- **Same scoring:** one parser and one scoring function for every run.

**Metrics** (see `blade_eval.py`):
- **F1**: a predicted box counts if it overlaps a ground-truth box of the same type with IoU ≥ 0.5. The headline number.
- **F1 box only**: the same, ignoring the defect type. Shows whether a model finds defects but names them wrongly.
- **Type accuracy**: whether the model's most frequent label is the image's defect type.
- **Parse rate**: answers that parsed as JSON. Low values mean the prompt or parser needs fixing, not that the model can't see.

**Running locally:** from `examples/wind-turbine-inspection/`, copy `.env.example` to `.env` and fill in `OPENAI_API_KEY`, plus `OPENAI_BASE_URL` if you use an OpenAI-compatible endpoint other than OpenAI (another provider's compatible API, or your own [LiteLLM](https://docs.litellm.ai/docs/simple_proxy) proxy to reach several providers with one key). Then start Jupyter with `uv run --group dev jupyter lab`. LFM3-VL runs on an Apple Silicon GPU (MPS), a CUDA GPU or the CPU.

## Configuration

Pick what to run and how many test images. Samples are nested: the first 10 images include the first one, and so on, so a larger run reuses the results of a smaller one. To compare runs, score them with the same `N_SAMPLES`.

In [ ]:
# What to run: "frontier", "lfm3_base" or "lfm3_finetune".
RUN = "lfm3_base"

# How many test images, spread evenly over the 9 defect types:
#   1     check that the notebook runs end to end
#   10    smoke test, about one image per defect type
#   90    comparison, 10 per defect type (oil leakage and lightning strikes have fewer)
#   None  the full test split, 1,372 images
N_SAMPLES = 100

# Frontier models to run: run name -> settings. Comment out the ones you don't want to run.
# `model` must match a model name on your endpoint (the run cell prints the ones it offers).
# `extra` is added to each request, e.g. {"reasoning_effort": "low"}.
FRONTIER_MODELS = {
    # "gemini": {"model": "gemini-3.1-pro", "box_format": "gemini_yxyx1000", "extra": {}},
    # Default reasoning: gpt-6-luna only accepts its default temperature while reasoning, so drop it.
    "gpt-6-luna": {"model": "gpt-6-luna", "box_format": "xyxy1000", "extra": {"temperature": None}},
    # "claude": {"model": "claude-opus-4-8", "box_format": "xyxy1000", "extra": {}},
}
DRY_RUN = False  # True prints the request for each frontier model without calling any API
MAX_WORKERS = 8  # parallel API requests; lower it if the proxy rate-limits

# LFM3-VL
LFM3_MODEL_ID = "LiquidAI/LFM3-VL-2B"
LFM3_ADAPTER_ID = None  # the LoRA adapter you pushed from lfm3_vl_grpo.ipynb (e.g. "your-username/lfm3-vl-blade-grpo"), for RUN = "lfm3_finetune"
LFM3_BOX_FORMAT = "xyxy1000"
DEVICE = None  # None picks "cuda", then "mps", then "cpu"
DTYPE = None  # None picks "bfloat16" on CUDA and "float32" on MPS/CPU (bfloat16 breaks on long inputs on MPS)
MAX_IMAGE_TOKENS = None  # None keeps the 640 px images at full resolution

# Optional: USD per million (input, output) tokens per run name, to estimate the cost per 1,000 images.
PRICES = {}

DATA_DIR = "../data"

## Setup

In [ ]:
import sys

if "google.colab" in sys.modules:  # locally, `uv run` installs everything from pyproject.toml
    %pip install -q openai pillow pandas matplotlib python-dotenv "transformers==5.20.0" "peft==0.21.2" "torchao==0.18.0" accelerate

import collections
import glob
import os
import random

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # finds ../.env when run from notebooks/
sys.path.insert(0, "..")  # blade_eval.py lives one directory up

import blade_eval as be

assert RUN in ("frontier", "lfm3_base", "lfm3_finetune"), f"unknown RUN {RUN!r}"

print(f"RUN: {RUN} | N_SAMPLES: {N_SAMPLES if N_SAMPLES is not None else 'all'}")
if RUN == "frontier":
    for run_name, settings in FRONTIER_MODELS.items():
        print(f"  {run_name}: {settings['model']} ({settings['box_format']})")
    print(f"  DRY_RUN: {DRY_RUN}")
else:
    print(f"  model: {LFM3_MODEL_ID} ({LFM3_BOX_FORMAT})")
    if RUN == "lfm3_finetune":
        print(f"  adapter: {LFM3_ADAPTER_ID}")
    device = DEVICE or be.pick_device()
    print(f"  device: {device} | dtype: {DTYPE or be.pick_dtype(device)} | max image tokens: {MAX_IMAGE_TOKENS or 'full resolution'}")

## Load the test images

The first run downloads the dataset (~480 MB) into `DATA_DIR`. Ground-truth boxes are drawn in green.

In [ ]:
root = be.download(DATA_DIR)
items = be.load_items(root, split="test", n=N_SAMPLES, exclude_cross_class_duplicates=True)
print(f"{len(items)} test images, {sum(len(item.boxes) for item in items)} defect boxes")
print(dict(collections.Counter(item.label for item in items)))

shown = list({item.label: item for item in reversed(items)}.values())[:10]
fig, axes = plt.subplots(1, len(shown), figsize=(4 * len(shown), 4), squeeze=False)
for ax, item in zip(axes[0], shown):
    ax.imshow(be.draw(item))
    ax.set_title(item.label)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Run

**Frontier** models are called through an OpenAI-compatible API, in parallel and with retries. Latency includes the network round trip.

**LFM3-VL** runs locally, one image at a time with greedy decoding and thinking off. The first run downloads the model (about 4 GB). Latency is measured on this machine, and each result row records the device.

Images already in a results file are skipped, so rerunning the cell resumes an interrupted run.

In [ ]:
if RUN == "frontier":
    from openai import OpenAI

    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"), base_url=os.environ.get("OPENAI_BASE_URL") or None)
    try:
        print("Models on the endpoint:", sorted(model.id for model in client.models.list()))
    except Exception as error:
        print(f"Couldn't list the proxy's models ({error}). Check OPENAI_API_KEY and OPENAI_BASE_URL in .env.")

    for run_name, settings in FRONTIER_MODELS.items():
        if DRY_RUN:
            request = be.api_request(items[0], settings["model"], settings["box_format"], settings["extra"])
            print(f"--- {run_name} ({settings['model']})")
            print(request["messages"][0]["content"][1]["text"])
        else:
            be.run_api(
                client,
                items,
                run_name,
                settings["model"],
                settings["box_format"],
                settings["extra"],
                max_workers=MAX_WORKERS,
            )
else:
    adapter_id = LFM3_ADAPTER_ID if RUN == "lfm3_finetune" else None
    assert RUN == "lfm3_base" or adapter_id, "Set LFM3_ADAPTER_ID to run the fine-tuned model."
    model, processor, device = be.load_local_model(LFM3_MODEL_ID, adapter_id, DEVICE, MAX_IMAGE_TOKENS, DTYPE)
    dtype = str(model.dtype).removeprefix("torch.")
    print(f"Loaded {LFM3_MODEL_ID}" + (f" + {adapter_id}" if adapter_id else "") + f" on {device} ({dtype})")

    # The precision is part of the run name, so float32 and bfloat16 results never mix in one file.
    precision = {"float32": "fp32", "bfloat16": "bf16"}.get(dtype, dtype)
    variant = "base" if RUN == "lfm3_base" else f"ft-{os.path.basename(adapter_id.rstrip('/'))}"
    run_name = f"lfm3-vl-2b-{variant}-{precision}"
    run_config = {
        "base_model": be.model_identity(LFM3_MODEL_ID),
        "adapter": be.model_identity(adapter_id) if adapter_id else None,
        "split": "test",
        "exclude_cross_class_duplicates": True,
        "max_image_tokens": MAX_IMAGE_TOKENS,
    }
    be.run_local(model, processor, items, run_name, LFM3_BOX_FORMAT, run_config=run_config)

## Results

Every run in `results/` is scored on the images of this `N_SAMPLES` setting, so runs are only compared on images they all answered. The confidence interval resamples images. Latency is per image: API latency includes the network, local latency depends on the device.

In [ ]:
ids = {item.item_id for item in items}
records = [r for path in sorted(glob.glob("results/*.jsonl")) for r in be.read_jsonl(path) if r["item_id"] in ids]
if records:
    display(be.summarize(records).round(3))
else:
    print("No results yet.")

Box F1 (with defect type) per defect class. With few images per class, treat these as rough.

In [ ]:
if records:
    display(be.per_class(records).round(2))

## Look at the predictions

Ground truth in green, predictions in red, for a few images per run. Each title shows true positives, false positives and false negatives.

In [ ]:
N_SHOW = 4
lookup = {item.item_id: item for item in items}
by_run = collections.defaultdict(list)
for record in records:
    by_run[record["model"]].append(record)

for run_name, run_records in by_run.items():
    sample = random.Random(0).sample(run_records, min(N_SHOW, len(run_records)))
    fig, axes = plt.subplots(1, len(sample), figsize=(5 * len(sample), 5), squeeze=False)
    for ax, record in zip(axes[0], sample):
        ax.imshow(be.draw(lookup[record["item_id"]], record))
        ax.set_title(f"{run_name}: {record['label']}\nTP {record['tp']}  FP {record['fp']}  FN {record['fn']}", fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

To read a single raw answer, e.g. when the parse rate is low:

In [ ]:
if records:
    print(records[0]["model"], records[0]["item_id"])
    print(records[0]["answer"])

## Cost

Tokens per image and, if `PRICES` is filled in, estimated USD per 1,000 images. Prices change often, so check them at the time of the run.

In [ ]:
if records:
    usage = pd.DataFrame(records).groupby("model")[["prompt_tokens", "completion_tokens"]].mean()
    usage.columns = ["prompt_tokens_per_image", "completion_tokens_per_image"]
    if PRICES:
        usage["usd_per_1k_images"] = [
            (row.prompt_tokens_per_image * PRICES[run][0] + row.completion_tokens_per_image * PRICES[run][1]) / 1000
            if run in PRICES
            else None
            for run, row in usage.iterrows()
        ]
    display(usage.round(4))